# Demo G: SGLang vs vLLM Prefix Matching

**Workshop Part 3** | LLM Inference at Scale | AI Engineering World's Fair 2026

**Platform:** Lightning.ai Studio (A100 GPU)

**Goal:** Prove that SGLang's RadixAttention finds longer common prefixes than
vLLM's hash-based approach. This matters for agents, multi-turn chat, and RAG.

## The Problem:
- **vLLM** hashes full prompt blocks. If two prompts differ at ANY point, no cache hit.
- **SGLang** builds a radix tree. Finds the LONGEST common prefix automatically.
- Multi-turn conversations share 90% of context but differ at the end. vLLM misses. SGLang hits.

## What we test:
| Workload | vLLM Expected | SGLang Expected |
|----------|--------------|------------------|
| Exact same prefix (identical system prompt) | Cache hit ✓ | Cache hit ✓ |
| Partial prefix (shared first 3 turns, different 4th) | Cache miss ✗ | Cache hit ✓ |

## Setup:
1. Same Lightning.ai Studio as Demo F
2. Install SGLang: `pip install 'sglang[all]'`
3. We test vLLM first, then SGLang, then compare


In [ ]:
# Setup: install deps + openai SDK
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'openai', 'torch', 'transformers', 'accelerate',
                       'matplotlib', 'requests', 'tqdm', 'numpy<2', 'scipy>=1.14'])

import time, requests
import matplotlib.pyplot as plt
from tqdm import tqdm
from openai import OpenAI
import concurrent.futures

# ─── Config ───
MODEL = 'mistralai/Mistral-7B-v0.1'
PORT = 8000
BASE_URL = f'http://localhost:{PORT}/v1'
client = OpenAI(base_url=BASE_URL, api_key='unused')

# ─── Benchmark Utility (shared with Demo F) ───
def benchmark_prefix(prompts, max_tokens=1, label=''):
    """Send prompts sequentially (to test prefix reuse per-request), measure TTFT + cache info."""
    request_details = []  # per-request: {ttft_ms, prompt_tokens, cached}
    
    for pi, prompt in enumerate(tqdm(prompts, desc=label)):
        t0 = time.perf_counter()
        first_token_time = None
        # Stream to get TTFT
        stream = client.completions.create(
            model=MODEL, prompt=prompt, max_tokens=max_tokens,
            temperature=0, stream=True
        )
        for chunk in stream:
            if first_token_time is None:
                first_token_time = time.perf_counter()
        ttft_ms = (first_token_time - t0) * 1000 if first_token_time else 0
        
        # Estimate cached tokens from TTFT (lower TTFT = more cached)
        request_details.append({
            'request': pi + 1,
            'ttft_ms': ttft_ms,
            'prompt_len': len(prompt.split()),  # rough word count
        })
    
    avg_ttft = sum(r['ttft_ms'] for r in request_details) / len(request_details)
    print(f'  [{label}] avg TTFT: {avg_ttft:.0f} ms')
    return request_details

print('Setup complete. Using OpenAI SDK for clean API calls.')


In [ ]:
# === Test Data: Two workload types to expose prefix caching differences ===

# --- Workload 1: EXACT prefix (identical system prompt, different queries) ---
# Both vLLM and SGLang should cache this perfectly after first request.
SYSTEM_PROMPT = (
    "You are a highly knowledgeable AI assistant specializing in machine learning, "
    "deep learning, natural language processing, and computer vision. You provide "
    "concise, accurate, and technically rigorous answers. Always cite relevant papers "
    "or frameworks when applicable. Focus on practical implementation details. "
    "Your responses should be suitable for senior ML engineers."
)  # ~100 tokens shared prefix

# 10 different user queries sharing the exact same system prompt
USER_QUERIES_EXACT = [
    "What is the difference between MHA and GQA?",
    "How does Flash Attention reduce memory usage?",
    "Explain continuous batching in 3 sentences.",
    "What is KV cache eviction and when is it needed?",
    "Compare tensor parallelism vs pipeline parallelism.",
    "What causes the memory wall in LLM inference?",
    "How does speculative decoding speed up generation?",
    "What is PagedAttention and why does it matter?",
    "Explain the roofline model for GPU workloads.",
    "What are the tradeoffs of quantization for serving?",
]

# Build exact-prefix prompts (system + user format)
EXACT_PREFIX_PROMPTS = [
    f"{SYSTEM_PROMPT}\n\nUser: {q}" for q in USER_QUERIES_EXACT
]

# --- Workload 2: PARTIAL prefix (multi-turn with shared history) ---
# Simulates multi-turn chat: first 3 turns identical, 4th turn differs.
# vLLM hash-based cache: hash of FULL prompt differs -> NO cache hit on partial.
# SGLang radix tree: finds longest matching prefix -> CACHE HIT on shared turns.
SHARED_TURNS = (
    "User: What is a transformer?\n"
    "Assistant: A transformer is a neural network architecture based on self-attention.\n\n"
    "User: How does attention work?\n"
    "Assistant: Attention computes weighted sums of value vectors using query-key similarity scores.\n\n"
    "User: What is the complexity?\n"
    "Assistant: Standard attention is O(n^2) in sequence length due to the full attention matrix.\n\n"
)  # ~150 tokens of shared conversation history

# 10 different 4th-turn queries appended to the shared history
TURN4_QUERIES = [
    "User: How can we reduce this to linear complexity?",
    "User: What is FlashAttention's approach to this?",
    "User: Does this apply to cross-attention too?",
    "User: How does KV cache help with this?",
    "User: What about sparse attention patterns?",
    "User: Is multi-query attention related to this?",
    "User: How does this affect batch inference?",
    "User: What is sliding window attention?",
    "User: Can we use approximate attention instead?",
    "User: How does ring attention distribute this?",
]

# Build partial-prefix prompts (shared history + unique 4th turn)
PARTIAL_PREFIX_PROMPTS = [
    f"{SHARED_TURNS}{q}" for q in TURN4_QUERIES
]

print(f"Exact prefix prompts: {len(EXACT_PREFIX_PROMPTS)} (shared system prompt)")
print(f"Partial prefix prompts: {len(PARTIAL_PREFIX_PROMPTS)} (shared 3-turn history)")
print(f"Exact prefix length: ~{len(SYSTEM_PROMPT.split())} words")
print(f"Partial prefix length: ~{len(SHARED_TURNS.split())} words")


## Step 1: vLLM with Prefix Caching (Baseline)

Start vLLM in a **separate terminal** with prefix caching enabled:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000 \
    --enable-prefix-caching
```

**How vLLM prefix caching works:** vLLM hashes fixed-size blocks of tokens. If the *entire* block sequence matches a previous request exactly, it reuses the cached KV values. This works great for identical prefixes but **fails on partial overlaps** where the divergence point falls mid-block or changes the hash chain.

Wait for the server to print "Uvicorn running on..." before proceeding.


In [ ]:
# === vLLM: Exact Prefix Test ===
# All 10 prompts share the IDENTICAL system prompt -> vLLM should cache after first request

# --- Warmup: prime the model with a DIFFERENT prompt (not from test set) ---
warmup_prompt = "Hello! This is a warmup request to compile CUDA kernels."
_ = benchmark_prefix(warmup_prompt, max_tokens=1)
print("Warmup complete (CUDA kernels compiled).")

# --- Cold run: first time seeing these prompts, no cache exists ---
print("\n--- vLLM Exact Prefix: COLD (no cache) ---")
vllm_exact_cold = benchmark_prompts(
    EXACT_PREFIX_PROMPTS, label="vLLM exact-cold", max_tokens=1
)
print(f"Mean TTFT (cold): {np.mean(vllm_exact_cold)*1000:.1f} ms")

# --- Warm run: same prompts again, cache should be populated ---
print("\n--- vLLM Exact Prefix: WARM (cache populated) ---")
vllm_exact_warm = benchmark_prompts(
    EXACT_PREFIX_PROMPTS, label="vLLM exact-warm", max_tokens=1
)
print(f"Mean TTFT (warm): {np.mean(vllm_exact_warm)*1000:.1f} ms")
print(f"Speedup: {np.mean(vllm_exact_cold)/np.mean(vllm_exact_warm):.2f}x")


In [ ]:
# === vLLM: Partial Prefix Test ===
# Prompts share first 3 turns but differ on 4th turn.
# vLLM hashes full block sequences -> partial overlap likely NOT cached.

# --- Cold run: first time seeing multi-turn prompts ---
print("--- vLLM Partial Prefix: COLD (no cache) ---")
vllm_partial_cold = benchmark_prompts(
    PARTIAL_PREFIX_PROMPTS, label="vLLM partial-cold", max_tokens=1
)
print(f"Mean TTFT (cold): {np.mean(vllm_partial_cold)*1000:.1f} ms")

# --- Warm run: same prompts again ---
# vLLM's hash-based cache may NOT help here because the full prompt hash differs
print("\n--- vLLM Partial Prefix: WARM (cache may miss) ---")
vllm_partial_warm = benchmark_prompts(
    PARTIAL_PREFIX_PROMPTS, label="vLLM partial-warm", max_tokens=1
)
print(f"Mean TTFT (warm): {np.mean(vllm_partial_warm)*1000:.1f} ms")
print(f"Speedup: {np.mean(vllm_partial_cold)/np.mean(vllm_partial_warm):.2f}x")
print("\nNote: Limited speedup expected -- vLLM hash misses on partial prefix overlap.")


## Step 2: SGLang with Radix Tree Caching

**Kill vLLM** in the terminal (`Ctrl+C`), then start SGLang:

```bash
python -m sglang.launch_server \
    --model-path mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --port 8000
```

**How SGLang prefix caching works:** SGLang uses a **radix tree** (trie) to store KV cache entries. It finds the **longest common prefix** between the new request and any previously cached request. This means:
- Exact prefix match → full cache hit (same as vLLM)  
- Partial prefix match → partial cache hit (SGLang wins)

The radix tree naturally handles multi-turn conversations where early turns are shared but later turns diverge.

Wait for "The server is fired up and ready to roll!" before proceeding.


In [ ]:
# === SGLang: Exact Prefix Test ===
# Same test as vLLM. SGLang should match vLLM's performance on exact prefixes.

# --- Warmup: different prompt to compile kernels ---
warmup_sglang = "Warmup request for SGLang server initialization."
_ = measure_ttft(warmup_sglang, max_tokens=1)
print("SGLang warmup complete.")

# --- Cold run: no cache yet ---
print("\n--- SGLang Exact Prefix: COLD (no cache) ---")
sglang_exact_cold = benchmark_prompts(
    EXACT_PREFIX_PROMPTS, label="SGLang exact-cold", max_tokens=1
)
print(f"Mean TTFT (cold): {np.mean(sglang_exact_cold)*1000:.1f} ms")

# --- Warm run: radix tree should have the shared prefix cached ---
print("\n--- SGLang Exact Prefix: WARM (cache populated) ---")
sglang_exact_warm = benchmark_prompts(
    EXACT_PREFIX_PROMPTS, label="SGLang exact-warm", max_tokens=1
)
print(f"Mean TTFT (warm): {np.mean(sglang_exact_warm)*1000:.1f} ms")
print(f"Speedup: {np.mean(sglang_exact_cold)/np.mean(sglang_exact_warm):.2f}x")


In [ ]:
# === SGLang: Partial Prefix Test ===
# Same multi-turn prompts. SGLang's radix tree finds longest common prefix.
# Expected: significant cache hit even though 4th turn differs.

# --- Cold run: first time seeing multi-turn prompts ---
print("--- SGLang Partial Prefix: COLD (no cache) ---")
sglang_partial_cold = benchmark_prompts(
    PARTIAL_PREFIX_PROMPTS, label="SGLang partial-cold", max_tokens=1
)
print(f"Mean TTFT (cold): {np.mean(sglang_partial_cold)*1000:.1f} ms")

# --- Warm run: radix tree should find longest matching prefix (first 3 turns) ---
print("\n--- SGLang Partial Prefix: WARM (radix tree match) ---")
sglang_partial_warm = benchmark_prompts(
    PARTIAL_PREFIX_PROMPTS, label="SGLang partial-warm", max_tokens=1
)
print(f"Mean TTFT (warm): {np.mean(sglang_partial_warm)*1000:.1f} ms")
print(f"Speedup: {np.mean(sglang_partial_cold)/np.mean(sglang_partial_warm):.2f}x")
print("\nNote: SGLang finds longest common prefix via radix tree -> cache hit on shared turns!")


## Per-Request Cache Analysis

The key insight: look at TTFT for EACH request, not just the average.
- Request 1: always slow (nothing cached yet)
- Request 2+: fast IF the engine found a cache hit

On the partial prefix workload, vLLM stays slow for ALL requests
(hash doesn't match), while SGLang drops after request 1 (radix tree finds overlap).


In [ ]:
# --- Per-Request TTFT Detail (shows cache hit pattern) ---
fig_detail, (ax_exact, ax_partial) = plt.subplots(1, 2, figsize=(12, 4))

# Exact prefix: per-request TTFT (should drop after first)
if 'vllm_exact' in dir() and 'sgl_exact' in dir():
    vllm_exact_ttfts = [r['ttft_ms'] for r in vllm_exact]
    sgl_exact_ttfts = [r['ttft_ms'] for r in sgl_exact]
    x_req = range(1, len(vllm_exact_ttfts) + 1)
    ax_exact.plot(x_req, vllm_exact_ttfts, 'o-', color='#2563eb', label='vLLM', markersize=5)
    ax_exact.plot(x_req, sgl_exact_ttfts, 's-', color='#16a34a', label='SGLang', markersize=5)
    ax_exact.set_xlabel('Request #')
    ax_exact.set_ylabel('TTFT (ms)')
    ax_exact.set_title('Exact Prefix: Per-Request TTFT', fontweight='bold')
    ax_exact.legend()
    ax_exact.spines['top'].set_visible(False)
    ax_exact.spines['right'].set_visible(False)

# Partial prefix: per-request TTFT (SGLang should be lower after first)
if 'vllm_partial' in dir() and 'sgl_partial' in dir():
    vllm_partial_ttfts = [r['ttft_ms'] for r in vllm_partial]
    sgl_partial_ttfts = [r['ttft_ms'] for r in sgl_partial]
    x_req2 = range(1, len(vllm_partial_ttfts) + 1)
    ax_partial.plot(x_req2, vllm_partial_ttfts, 'o-', color='#2563eb', label='vLLM', markersize=5)
    ax_partial.plot(x_req2, sgl_partial_ttfts, 's-', color='#16a34a', label='SGLang', markersize=5)
    ax_partial.set_xlabel('Request #')
    ax_partial.set_ylabel('TTFT (ms)')
    ax_partial.set_title('Partial Prefix: Per-Request TTFT\n(SGLang reuses partial, vLLM cannot)', fontweight='bold')
    ax_partial.legend()
    ax_partial.spines['top'].set_visible(False)
    ax_partial.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Left: Both engines cache exact prefixes (request 2+ is fast).')
print('Right: Only SGLang caches partial prefixes (vLLM stays slow for all).')


In [ ]:
# === Comparison Chart: vLLM vs SGLang Prefix Caching ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: Exact prefix comparison ---
ax1 = axes[0]
categories_exact = ["Cold\n(no cache)", "Warm\n(cached)"]
vllm_exact_means = [np.mean(vllm_exact_cold)*1000, np.mean(vllm_exact_warm)*1000]
sglang_exact_means = [np.mean(sglang_exact_cold)*1000, np.mean(sglang_exact_warm)*1000]

x_exact = np.arange(len(categories_exact))  # Bar positions
width = 0.35  # Bar width

bars1 = ax1.bar(x_exact - width/2, vllm_exact_means, width, label="vLLM", color="#dbeafe", edgecolor="#000")
bars2 = ax1.bar(x_exact + width/2, sglang_exact_means, width, label="SGLang", color="#dcfce7", edgecolor="#000")

ax1.set_xlabel("Run Type")
ax1.set_ylabel("Mean TTFT (ms)")
ax1.set_title("Exact Prefix: Both Engines Cache Well")
ax1.set_xticks(x_exact)
ax1.set_xticklabels(categories_exact)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar in bars1 + bars2:
    height = bar.get_height()
    ax1.annotate(f"{height:.0f}", xy=(bar.get_x() + bar.get_width()/2, height),
                 xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

# --- Right panel: Partial prefix comparison (the key insight) ---
ax2 = axes[1]
categories_partial = ["Cold\n(no cache)", "Warm\n(cached)"]
vllm_partial_means = [np.mean(vllm_partial_cold)*1000, np.mean(vllm_partial_warm)*1000]
sglang_partial_means = [np.mean(sglang_partial_cold)*1000, np.mean(sglang_partial_warm)*1000]

x_partial = np.arange(len(categories_partial))  # Bar positions

bars3 = ax2.bar(x_partial - width/2, vllm_partial_means, width, label="vLLM", color="#dbeafe", edgecolor="#000")
bars4 = ax2.bar(x_partial + width/2, sglang_partial_means, width, label="SGLang", color="#dcfce7", edgecolor="#000")

ax2.set_xlabel("Run Type")
ax2.set_ylabel("Mean TTFT (ms)")
ax2.set_title("Partial Prefix: SGLang Radix Tree Wins")
ax2.set_xticks(x_partial)
ax2.set_xticklabels(categories_partial)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar in bars3 + bars4:
    height = bar.get_height()
    ax2.annotate(f"{height:.0f}", xy=(bar.get_x() + bar.get_width()/2, height),
                 xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.suptitle("Demo G: Prefix Caching — vLLM (Hash) vs SGLang (Radix Tree)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("demo_g_prefix_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved: demo_g_prefix_comparison.png")


## Key Takeaways: When to Use SGLang over vLLM for Prefix Caching

| Workload | vLLM | SGLang | Winner |
|----------|------|--------|--------|
| Identical system prompts | ✅ Cached (hash match) | ✅ Cached (radix match) | Tie |
| Multi-turn with shared history | ❌ Miss (hash of full prompt differs) | ✅ Hit (longest prefix match) | **SGLang** |
| RAG with shared retrieved docs | ❌ Miss | ✅ Partial hit | **SGLang** |
| Completely unique prompts | ❌ No benefit | ❌ No benefit | Tie |

### The Core Insight

**vLLM** hashes fixed-size token blocks. If the full block sequence matches exactly, cache hit. Any divergence breaks the hash chain.

**SGLang** uses a **radix tree** (trie) that finds the longest matching prefix character-by-character (token-by-token). Partial matches still save compute for the matched portion.

### When SGLang's Radix Tree Matters Most

1. **Multi-turn agents**: Conversation history grows but early turns are shared across requests
2. **RAG pipelines**: Retrieved context partially overlaps between queries  
3. **Few-shot prompting**: Examples are shared but the final query differs
4. **Code completion**: File context is shared but cursor position changes

### When It Doesn't Matter

- Fixed system prompts with short user queries (both engines cache fine)
- Completely unique prompts (nothing to cache)
- Single-request workloads (no reuse opportunity)
